# Unified Topology NCA & ARC Manifold Reasoning
High-leverage training suite with autonomous manifold evolution, financial tasks, and ARC solving.

In [ ]:
%%bash
echo "Fixing environment..."
pip install --no-cache-dir --force-reinstall torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2+cu117 --index-url https://download.pytorch.org/whl/cu117
pip install --upgrade kaggle kagglehub wandb pandas numpy

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_453cfb028676f79df571e5b2a8ee6afd'
os.environ['KAGGLE_USERNAME'] = 'hichambedrani'
os.environ['KAGGLE_KEY'] = 'KGAT_453cfb028676f79df571e5b2a8ee6afd'

os.makedirs('kaggle_data', exist_ok=True)
!mkdir -p kaggle_data/sec_financials
!kaggle datasets download -d securities-exchange-commission/financial-statement-extracts -p kaggle_data/sec_financials --unzip

In [ ]:
%%writefile monitoring_utils.py
import torch
import numpy as np
import os

def calculate_leverage(accuracy, ber, weights_norm):
    efficiency = 1.0 / (1.0 + weights_norm)
    leverage = (accuracy * (1.0 - ber) * efficiency) * 100
    return leverage

def monitor_model_health(model):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    return total_norm

class WandbLogger:
    def __init__(self, project_name="topo-neural", config=None):
        try:
            import wandb
            self.wandb = wandb
            if os.environ.get('WANDB_API_KEY'):
                self.wandb.init(project=project_name, config=config)
                self.enabled = True
            else:
                print("WANDB_API_KEY not found. Wandb logging disabled.")
                self.enabled = False
        except ImportError:
            print("Wandb not installed. Logging disabled.")
            self.enabled = False

    def log(self, metrics):
        if self.enabled:
            self.wandb.log(metrics)

    def finish(self):
        if self.enabled:
            self.wandb.finish()


In [ ]:
%%writefile data_utils.py
import json
import torch
from torch.utils.data import Dataset, DataLoader

class StratosCoTDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                instruction = data['instruction']
                response = data['response']

                # Extract input for prediction (last line of instruction)
                input_str = instruction.split(':')[-1].strip()

                # Extract target from boxed answer
                if '\\boxed{' in response:
                    target_str = response.split('\\boxed{')[-1].split('}')[0].strip()
                else:
                    continue

                if len(input_str) == 8 and len(target_str) == 8:
                    input_bits = [int(b) for b in input_str]
                    target_bits = [int(b) for b in target_str]
                    self.samples.append((
                        torch.tensor(input_bits, dtype=torch.float32),
                        torch.tensor(target_bits, dtype=torch.float32)
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def get_dataloader(file_path, batch_size=32):
    dataset = StratosCoTDataset(file_path)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

if __name__ == "__main__":
    loader = get_dataloader('./kaggle_data/stratoscot/augmented_train.jsonl')
    x, y = next(iter(loader))
    print(f"Batch X shape: {x.shape}")
    print(f"Batch Y shape: {y.shape}")


In [ ]:
%%writefile kaggle_utils.py




import os
import zipfile
import time
from kaggle.api.kaggle_api_extended import KaggleApi

def download_dataset(dataset, path, retries=3):
    if os.path.exists(path) and len(os.listdir(path)) > 0:
        print(f"Data already exists at {path}. Skipping download.")
        return True

    print(f"Downloading dataset {dataset} to {path}...")
    api = KaggleApi()
    api.authenticate()
    os.makedirs(path, exist_ok=True)

    for i in range(retries):
        try:
            api.dataset_download_files(dataset, path=path, unzip=True)
            print(f"Download of {dataset} complete.")
            return True
        except Exception as e:
            print(f"Attempt {i+1} failed to download {dataset}: {e}")
            if i < retries - 1:
                time.sleep(5)
            else:
                raise e
    return False

def download_manifold_data():
    return download_dataset('hichambedrani/stratos-manifold-v4', './kaggle_data/stratos_manifold')

def download_all_resources():
    resources = [
        ('hichambedrani/stratos-manifold-v4', './kaggle_data/stratos_manifold'),
        ('hichambedrani/stratoscot', './kaggle_data/stratoscot'),
        ('hichambedrani/stratos-omega-manifold-v3', './kaggle_data/omega_manifold'),
        ('hichambedrani/fso-manifold', './kaggle_data/fso_manifold'),
        ('hichambedrani/precision-system-v3-data', './kaggle_data/precision_data')
    ]
    for ds, path in resources:
        download_dataset(ds, path)

class KaggleSearch:
    """
    Search utility for Kaggle Datasets and Models.
    """
    def __init__(self):
        self.api = KaggleApi()
        self.api.authenticate()

    def search_datasets(self, query):
        print(f"Searching for datasets matching: {query}")
        datasets = self.api.dataset_list(search=query)
        for ds in datasets:
            print(f"Dataset: {ds.ref} | Title: {ds.title}")
        return datasets

    def search_models(self, query):
        print(f"Searching for models matching: {query}")
        try:
            models = self.api.model_list(search=query)
            for model in models:
                print(f"Model: {model.ownerSlug}/{model.slug} | Title: {model.title}")
            return models
        except AttributeError:
            print("Model search not supported in this Kaggle API version.")
            return []

    def discover_and_download_resources(self, query, base_path='./kaggle_data/discovered'):
        print(f"Discovering and downloading resources for: {query}")
        datasets = self.search_datasets(query)
        downloaded_paths = []
        for ds in datasets[:3]: # Limit to top 3
            path = os.path.join(base_path, ds.ref.replace('/', '_'))
            if download_dataset(ds.ref, path):
                downloaded_paths.append(path)
        return downloaded_paths

if __name__ == "__main__":
    search = KaggleSearch()
    search.search_datasets("manifold")


In [ ]:
%%writefile kaggle_hub_manager.py
import kagglehub
import os
import shutil
import torch
import json

class KaggleHubManager:
    """
    Manages model interactions with the Kaggle Model Hub using kagglehub.
    """
    def __init__(self, model_handle=None):
        self.model_handle = model_handle

    def download_model(self, handle=None):
        handle = handle or self.model_handle
        if not handle:
            raise ValueError("No model handle provided.")

        print(f"Downloading model from Kaggle Hub: {handle}...")
        path = kagglehub.model_download(handle)
        print(f"Model downloaded to: {path}")
        return path

    def upload_model_version(self, handle, local_model_dir, version_notes="New model version"):
        """
        Uploads a new version of a model to Kaggle Model Hub.
        handle: 'owner/model/framework/variation'
        """
        if not os.path.exists(local_model_dir):
            raise FileNotFoundError(f"Local model directory {local_model_dir} does not exist.")

        print(f"Uploading model version to {handle} from {local_model_dir}...")
        try:
            path = kagglehub.model_upload(handle, local_model_dir, version_notes=version_notes)
            print(f"Model successfully uploaded to {handle}")
            return path
        except Exception as e:
            print(f"Failed to upload model: {e}")
            return None

def save_and_push_to_hub(model, optimizer, epoch, metrics, handle, local_dir='checkpoint'):
    """
    Helper to save a checkpoint locally and push it to Kaggle Hub.
    """
    os.makedirs(local_dir, exist_ok=True)
    checkpoint_path = os.path.join(local_dir, 'model.pt')
    torch_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics
    }
    torch.save(torch_state, checkpoint_path)

    with open(os.path.join(local_dir, 'metadata.json'), 'w') as f:
        json.dump(metrics, f)

    manager = KaggleHubManager()
    manager.upload_model_version(handle, local_dir, version_notes=f"Epoch {epoch} checkpoint")

if __name__ == "__main__":
    pass


In [ ]:
%%writefile weights_loader.py
import os
import numpy as np
import torch
from kaggle_utils import download_all_resources

class ManifoldLoader:
    """
    Enhanced utility to load weights from multiple Stratos/Omega/FSO Manifold datasets.
    Includes strict shape validation.
    """
    def __init__(self, source='stratos', directory=None):
        # download_all_resources()

        sources = {
            'stratos': 'kaggle_data/stratos_manifold',
            'omega': 'kaggle_data/omega_manifold',
            'fso': 'kaggle_data/fso_manifold'
        }

        base_dir = directory or sources.get(source)
        if not base_dir:
            raise ValueError(f"Unknown source: {source}")

        # Find the actual directory containing .npy files
        self.directory = self._find_npy_dir(base_dir)
        print(f"[{source.upper()} LOADER] Using directory: {self.directory}")

        self.weight_files = sorted([f for f in os.listdir(self.directory) if f.startswith('weight_')])
        self.lib_files = sorted([f for f in os.listdir(self.directory) if f.startswith('lib_')])
        self.ptr = 0

    def _find_npy_dir(self, start_path):
        if not os.path.exists(start_path):
             raise FileNotFoundError(f"Directory {start_path} does not exist.")
        for root, dirs, files in os.walk(start_path):
            if any(f.endswith('.npy') for f in files):
                return root
        raise FileNotFoundError(f"No .npy files found in {start_path}")

    def load_weights(self, num_weights, edge_dim=32, node_dim=32):
        end = min(self.ptr + num_weights, len(self.weight_files))
        actual_num = end - self.ptr

        weights = []
        target_size = edge_dim * node_dim

        for i in range(self.ptr, end):
            file_path = os.path.join(self.directory, self.weight_files[i])
            try:
                data = np.load(file_path)
            except Exception as e:
                raise IOError(f"Failed to load weight file {file_path}: {e}")

            # Flatten then reshape to fit requested dimensions
            flat_data = data.flatten()

            if flat_data.size < target_size:
                print(f"Warning: Data size {flat_data.size} in {self.weight_files[i]} is less than target {target_size}. Padding with zeros.")
                padded = np.zeros(target_size)
                padded[:flat_data.size] = flat_data
                reshaped = padded.reshape(edge_dim, node_dim)
            else:
                if flat_data.size > target_size:
                    print(f"Warning: Data size {flat_data.size} in {self.weight_files[i]} exceeds target {target_size}. Truncating.")
                reshaped = flat_data[:target_size].reshape(edge_dim, node_dim)
            weights.append(reshaped)

        self.ptr = end

        # If not enough weights, instead of random, we can raise error or pad.
        # Original code used random padding. Keeping it but with a message.
        if len(weights) < num_weights:
            print(f"Warning: Requested {num_weights} weights but only found {len(weights)}. Padding with random.")
            padding = [np.random.randn(edge_dim, node_dim) for _ in range(num_weights - len(weights))]
            weights.extend(padding)

        return torch.tensor(np.array(weights), dtype=torch.float32)

    def load_library(self, num_libs, edge_dim=32, node_dim=32):
        if num_libs > len(self.lib_files):
            print(f"Warning: Requested {num_libs} libraries but only found {len(self.lib_files)}.")
            num_libs = len(self.lib_files)

        libs = []
        target_size = edge_dim * node_dim
        for i in range(num_libs):
            file_path = os.path.join(self.directory, self.lib_files[i])
            try:
                data = np.load(file_path)
            except Exception as e:
                raise IOError(f"Failed to load library file {file_path}: {e}")

            flat_data = data.flatten()
            if flat_data.size < target_size:
                padded = np.zeros(target_size)
                padded[:flat_data.size] = flat_data
                reshaped = padded.reshape(edge_dim, node_dim)
            else:
                reshaped = flat_data[:target_size].reshape(edge_dim, node_dim)
            libs.append(reshaped)

        return torch.tensor(np.array(libs), dtype=torch.float32)


In [ ]:
%%writefile sheaf_nn.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SheafDiffusionLayer(nn.Module):
    def __init__(self, num_nodes, edges, node_dim, edge_dim, alpha=0.01):
        super(SheafDiffusionLayer, self).__init__()
        self.num_nodes = num_nodes
        self.num_edges = len(edges)
        self.d = node_dim
        self.de = edge_dim
        self.alpha = alpha

        self.W_maps = nn.Parameter(torch.randn(2 * self.num_edges, self.de, self.d))
        self.register_buffer('edge_index', torch.tensor(edges).t().contiguous())

    def load_from_manifold(self, loader):
        """
        Initializes W_maps using weights from the ManifoldLoader.
        """
        manifold_weights = loader.load_weights(2 * self.num_edges, edge_dim=self.de, node_dim=self.d)
        if manifold_weights.shape == self.W_maps.shape:
            self.W_maps.data.copy_(manifold_weights)
            print(f"Successfully loaded {2 * self.num_edges} manifold weights into W_maps.")
        else:
            print(f"Warning: Manifold weight shape {manifold_weights.shape} does not match W_maps shape {self.W_maps.shape}.")

    def forward(self, H):
        batch_size = H.size(0)
        W_src = self.W_maps[0::2]
        W_dst = self.W_maps[1::2]
        u_idx = self.edge_index[0]
        v_idx = self.edge_index[1]

        H_u = H[:, u_idx, :]
        H_v = H[:, v_idx, :]

        proj_u = torch.matmul(W_src, H_u.unsqueeze(-1)).squeeze(-1)
        proj_v = torch.matmul(W_dst, H_v.unsqueeze(-1)).squeeze(-1)

        Z = proj_u - proj_v

        grad_u = torch.matmul(W_src.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)
        grad_v = torch.matmul(W_dst.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)

        Delta_H = torch.zeros_like(H)
        expanded_u_idx = u_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_u_idx, grad_u)
        expanded_v_idx = v_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_v_idx, -grad_v)

        H_new = H - self.alpha * Delta_H
        return F.relu(H_new)

    def project_to_stiefel(self):
        with torch.no_grad():
            W = self.W_maps
            WtW = torch.matmul(W.transpose(-1, -2), W)
            e, v = torch.linalg.eigh(WtW)
            e_inv_sqrt = torch.diag_embed(1.0 / torch.sqrt(torch.clamp(e, min=1e-6)))
            WtW_inv_sqrt = torch.matmul(torch.matmul(v, e_inv_sqrt), v.transpose(-1, -2))
            W_new = torch.matmul(W, WtW_inv_sqrt)
            self.W_maps.copy_(W_new)

class SheafNCALayer(SheafDiffusionLayer):
    """
    Learned local update rule based on Sheaf residuals.
    Instead of simple diffusion, uses an MLP to compute the update.
    """
    def __init__(self, num_nodes, edges, node_dim, edge_dim, hidden_dim=16):
        super(SheafNCALayer, self).__init__(num_nodes, edges, node_dim, edge_dim)
        # MLP takes [local_feature, sheaf_residual]
        self.mlp = nn.Sequential(
            nn.Linear(node_dim + node_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim),
        )

    def load_from_manifold(self, loader):
        """
        Initializes both W_maps and the internal MLP using manifold weights.
        """
        super().load_from_manifold(loader)

        # Load MLP weights from 'lib' files
        lib_weights = loader.load_library(2, edge_dim=1, node_dim=1024) # Placeholder for more complex mapping
        print("Note: MLP manifold integration is using available lib tensors.")

    def forward(self, H):
        batch_size = H.size(0)
        W_src = self.W_maps[0::2]
        W_dst = self.W_maps[1::2]
        u_idx = self.edge_index[0]
        v_idx = self.edge_index[1]

        H_u = H[:, u_idx, :]
        H_v = H[:, v_idx, :]

        proj_u = torch.matmul(W_src, H_u.unsqueeze(-1)).squeeze(-1)
        proj_v = torch.matmul(W_dst, H_v.unsqueeze(-1)).squeeze(-1)

        Z = proj_u - proj_v

        grad_u = torch.matmul(W_src.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)
        grad_v = torch.matmul(W_dst.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)

        Delta_H = torch.zeros_like(H)
        expanded_u_idx = u_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_u_idx, grad_u)
        expanded_v_idx = v_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_v_idx, -grad_v)

        # Local update: concatenate current feature and aggregated sheaf residual
        concat_feat = torch.cat([H, Delta_H], dim=-1)
        update = self.mlp(concat_feat)

        H_new = H + update
        return F.relu(H_new)

class DeepSheafNetwork(nn.Module):
    def __init__(self, num_nodes, edges, node_dim, edge_dim, num_layers=3, alpha=0.01, layer_type='diffusion'):
        super(DeepSheafNetwork, self).__init__()
        self.layer_type = layer_type
        if layer_type == 'diffusion':
            self.layers = nn.ModuleList([
                SheafDiffusionLayer(num_nodes, edges, node_dim, edge_dim, alpha=alpha)
                for _ in range(num_layers)
            ])
        elif layer_type == 'nca':
            self.layers = nn.ModuleList([
                SheafNCALayer(num_nodes, edges, node_dim, edge_dim)
                for _ in range(num_layers)
            ])

    def load_from_manifold(self, loader):
        """
        Initializes all layers using weights from the ManifoldLoader.
        """
        for i, layer in enumerate(self.layers):
            print(f"Loading manifold weights for layer {i}...")
            layer.load_from_manifold(loader)

    def forward(self, H):
        for layer in self.layers:
            H = layer(H)
        return H

    def project_all_to_stiefel(self):
        for layer in self.layers:
            layer.project_to_stiefel()


In [ ]:
%%writefile topo_torch.py
import torch

def relaxed_euler_torch(grid):
    """
    Continuous relaxation of the Euler Characteristic in PyTorch.
    Supports backpropagation.
    grid: Tensor of shape [Batch, H, W] with values in [0, 1]
    """
    # V: Sum of vertices
    V = torch.sum(grid, dim=(1, 2))

    # E_h: Horizontal edges
    E_h = torch.sum(grid[:, :, :-1] * grid[:, :, 1:], dim=(1, 2))

    # E_v: Vertical edges
    E_v = torch.sum(grid[:, :-1, :] * grid[:, 1:, :], dim=(1, 2))

    E = E_h + E_v

    # F: Faces (quads)
    F = torch.sum(grid[:, :-1, :-1] * grid[:, :-1, 1:] * grid[:, 1:, :-1] * grid[:, 1:, 1:], dim=(1, 2))

    return V - E + F

def compute_euler_binary_torch(grid_binary):
    """
    Exact Euler Characteristic for binary tensors in PyTorch.
    grid_binary: Tensor of shape [Batch, H, W] with values in {0, 1}
    """
    V = torch.sum(grid_binary, dim=(1, 2))

    # Use logical_and for exactness, though multiplication works for binary
    E_h = torch.sum(grid_binary[:, :, :-1] * grid_binary[:, :, 1:], dim=(1, 2))
    E_v = torch.sum(grid_binary[:, :-1, :] * grid_binary[:, 1:, :], dim=(1, 2))
    E = E_h + E_v

    F = torch.sum(grid_binary[:, :-1, :-1] * grid_binary[:, :-1, 1:] * grid_binary[:, 1:, :-1] * grid_binary[:, 1:, 1:], dim=(1, 2))

    return V - E + F


In [ ]:
%%writefile spectral_topo.py
import torch

def assemble_coboundary_matrix(num_nodes, edge_index, W_maps, de, d):
    """
    Assembles the Coboundary Matrix D_F as a dense tensor.
    Vectorized implementation.
    W_maps: [2 * E, de, d]
    edge_index: [2, E]
    Returns: D_F of shape [E * de, V * d]
    """
    num_edges = edge_index.size(1)
    device = W_maps.device
    dtype = W_maps.dtype

    D_F = torch.zeros(num_edges * de, num_nodes * d, device=device, dtype=dtype)

    # Indices for row-wise blocks
    row_starts = torch.arange(num_edges, device=device) * de

    # Source blocks (W_u)
    u_idx = edge_index[0] # [E]
    u_col_starts = u_idx * d # [E]

    # Destination blocks (-W_v)
    v_idx = edge_index[1] # [E]
    v_col_starts = v_idx * d # [E]

    W_src = W_maps[0::2] # [E, de, d]
    W_dst = W_maps[1::2] # [E, de, d]

    # Create grid of offsets within each block
    ii, jj = torch.meshgrid(torch.arange(de, device=device), torch.arange(d, device=device), indexing='ij')

    # Expand to all edges
    row_indices = (row_starts.view(-1, 1, 1) + ii.view(1, de, d)).view(-1)
    col_indices_src = (u_col_starts.view(-1, 1, 1) + jj.view(1, de, d)).view(-1)
    col_indices_dst = (v_col_starts.view(-1, 1, 1) + jj.view(1, de, d)).view(-1)

    # Flattened indices for 1D scatter
    stride = num_nodes * d
    idx_src = row_indices * stride + col_indices_src
    idx_dst = row_indices * stride + col_indices_dst

    D_F.view(-1).scatter_add_(0, idx_src, W_src.reshape(-1))
    D_F.view(-1).scatter_add_(0, idx_dst, -W_dst.reshape(-1))

    return D_F

def compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, W_maps, de, d):
    """
    Computes the second smallest eigenvalue of the Sheaf Laplacian.
    This serves as a differentiable proxy for connectivity/alignment.
    """
    D_F = assemble_coboundary_matrix(num_nodes, edge_index, W_maps, de, d)
    L_F = torch.matmul(D_F.t(), D_F) # [V*d, V*d]

    # Compute eigenvalues
    eigenvalues = torch.linalg.eigvalsh(L_F)

    # Return the second smallest eigenvalue
    if eigenvalues.numel() > 1:
        return eigenvalues[1]
    else:
        return eigenvalues[0]

def spectral_connectivity_loss(num_nodes, edge_index, W_maps, de, d, target_gap=0.1):
    """
    Loss that encourages the Sheaf Laplacian to have a spectral gap.
    """
    gap = compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, W_maps, de, d)
    return torch.relu(target_gap - gap)


In [ ]:
%%writefile train_high_leverage.py
import torch
import torch.nn as nn
import torch.optim as optim
from sheaf_nn import DeepSheafNetwork
from weights_loader import ManifoldLoader
from data_utils import get_dataloader
from monitoring_utils import calculate_leverage, monitor_model_health, WandbLogger
from kaggle_hub_manager import save_and_push_to_hub
from topo_torch import relaxed_euler_torch
from spectral_topo import compute_sheaf_laplacian_spectral_gap
import json
import os

def train(dry_run=False, use_topo_loss=True, num_epochs=100000, checkpoint_freq=500):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Large Scale High-Leverage Training (Target: {num_epochs} Epochs) ---")
    print(f"Device: {device}")

    num_nodes = 8
    edges = [(i, (i + 1) % num_nodes) for i in range(num_nodes)]
    for i in range(num_nodes):
        edges.append((i, (i + 2) % num_nodes))

    edge_index = torch.tensor(edges).t().to(device)

    node_dim = 32
    edge_dim = 32
    num_layers = 12

    config = {
        "num_nodes": num_nodes, "node_dim": node_dim, "edge_dim": edge_dim,
        "num_layers": num_layers, "lr": 0.0001, "weight_decay": 0.01,
        "topo_weight": 0.1, "spectral_weight": 0.05, "target_epochs": num_epochs
    }

    try:
        model = DeepSheafNetwork(num_nodes, edges, node_dim, edge_dim, num_layers=num_layers, layer_type='nca').to(device)
        if device.type == 'cuda':
            dummy = torch.randn(1, num_nodes, node_dim).to(device)
            model(dummy)
            print("GPU model verification successful.")
    except Exception as e:
        if 'no kernel image' in str(e) or 'CUDA error' in str(e):
            print(f"GPU error: {e}. Falling back to CPU.")
            device = torch.device('cpu')
            edge_index = edge_index.to(device)
            model = DeepSheafNetwork(num_nodes, edges, node_dim, edge_dim, num_layers=num_layers, layer_type='nca').to(device)
        else:
            raise e

    logger = WandbLogger(project_name="topo-neural-ultra-long", config=config)
    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == 'cuda'))

    print("Initializing from Multiple Manifolds...")
    try:
        stratos_loader = ManifoldLoader(source='stratos')
        omega_loader = ManifoldLoader(source='omega')
        for i, layer in enumerate(model.layers):
            loader = stratos_loader if i % 2 == 0 else omega_loader
            layer.load_from_manifold(loader)
    except Exception as e:
        print(f"Initialization from manifold failed: {e}. Using random weights.")

    output_head = nn.Sequential(
        nn.Linear(node_dim, 128), nn.LayerNorm(128), nn.ReLU(), nn.Linear(128, 8)
    ).to(device)

    optimizer = optim.AdamW(list(model.parameters()) + list(output_head.parameters()), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100) # Longer cycle for 100k
    criterion = nn.BCEWithLogitsLoss()

    report_file = 'TRAINING_REPORT.jsonl'

    try:
        dataloader = get_dataloader('./kaggle_data/stratoscot/augmented_train.jsonl', batch_size=256)
    except Exception as e:
        print(f"Dataloader failed: {e}. Using dummy data.")
        dataloader = [(torch.randn(10, 8), torch.randint(0, 2, (10, 8)).float())]

    actual_epochs = num_epochs if not dry_run else 2
    best_leverage = -1
    early_stop_counter = 0

    print("Starting Epochs...")
    for epoch in range(actual_epochs):
        model.train()
        total_loss, total_ber, correct, total = 0, 0, 0, 0

        for batch_idx, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            autocast_device = device.type if device.type in ['cuda', 'cpu'] else 'cpu'
            with torch.amp.autocast(autocast_device, enabled=(device.type == 'cuda')):
                H = x.unsqueeze(-1).repeat(1, 1, node_dim)
                H_out = model(H)
                logits = output_head(H_out).mean(dim=1)
                main_loss = criterion(logits, y)

                topo_loss = 0
                if use_topo_loss:
                    grid_probs = torch.sigmoid(logits).view(-1, 2, 4)
                    chi = relaxed_euler_torch(grid_probs)
                    topo_loss = torch.mean((chi - 1.0)**2)
                    last_layer = model.layers[-1]
                    gap = compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, last_layer.W_maps, last_layer.de, last_layer.d)
                    spectral_loss = torch.relu(0.1 - gap)
                    loss = main_loss + config["topo_weight"] * topo_loss + config["spectral_weight"] * spectral_loss
                else:
                    loss = main_loss

            if torch.isnan(loss):
                print(f"NaN loss detected at epoch {epoch}. Stopping.")
                return

            scaler.scale(loss).backward()
            grad_norm = monitor_model_health(model)
            if device.type == 'cuda': scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).all(dim=1).sum().item()
            total += x.size(0)
            total_ber += torch.mean((preds != y).float()).item()
            if dry_run and batch_idx >= 2: break

        scheduler.step()
        avg_acc = correct / total if total > 0 else 0
        avg_ber = total_ber / len(dataloader)
        with torch.no_grad():
            w_norm = sum(p.norm(2).item() for p in model.parameters())
            leverage = calculate_leverage(avg_acc, avg_ber, w_norm)

        metrics = {"epoch": epoch, "loss": total_loss / len(dataloader), "accuracy": avg_acc, "ber": avg_ber, "leverage": leverage, "grad_norm": grad_norm}
        logger.log(metrics)

        if epoch % 10 == 0 or dry_run:
            with open(report_file, 'a') as f: f.write(json.dumps(metrics) + '\n')
            print(f"Epoch {epoch}: Loss={metrics['loss']:.4f}, Acc={avg_acc:.2%}, Leverage={leverage:.4f}")

        # Checkpointing
        if leverage > best_leverage:
            best_leverage = leverage
            early_stop_counter = 0
            if not dry_run:
                handle = os.environ.get('KAGGLE_MODEL_HANDLE')
                if handle: save_and_push_to_hub(model, optimizer, epoch, metrics, handle, local_dir='best_checkpoint')
        else:
            early_stop_counter += 1

        if epoch % checkpoint_freq == 0 and not dry_run:
            handle = os.environ.get('KAGGLE_MODEL_HANDLE')
            if handle: save_and_push_to_hub(model, optimizer, epoch, metrics, handle, local_dir=f'epoch_{epoch}_checkpoint')

        if early_stop_counter > 5000 and not dry_run: # Patience for 100k epochs
            print(f"Early stopping at epoch {epoch}")
            break

    logger.finish()
    print("High-Leverage Model Training Complete.")

if __name__ == "__main__":
    train(dry_run=True)


In [ ]:
%%writefile autonomous_manifold_v3.py
import hashlib
import math
import random
import json
import os

class ManifoldConfig:
    def __init__(self, config_file=None):
        self.config = self._load_default_config()
        if config_file and os.path.exists(config_file):
            self._load_config_from_file(config_file)

    def _load_default_config(self):
        return {
            "initial_mass": 0.1,
            "will_strength_multiplier": 1.5,
            "q_weights": {
                "Creativity": 0.08,
                "Logic": 0.18,
                "Density": 0.16,
                "Realization": 0.20,
                "Entropy": 0.12
            },
            "destructive_coordinates": {
                "entropy_collapse": 0.8,
                "logic_override": 0.9,
                "density_singularity": 0.95
            },
            "learning_rate": 0.25,
            "target_state": {"Creativity": 0.9, "Logic": 0.9, "Density": 0.7, "Realization": 0.98, "Entropy": 0.05},
            "agent_initial_masses": {
                "Creativity_Agent": 0.15,
                "Logic_Agent": 0.15,
                "Density_Agent": 0.2,
                "Realization_Agent": 0.12,
                "Entropy_Agent": 0.08
            },
            "keyword_influences": {
                "creative": ("Creativity", 0.5),
                "logic": ("Logic", 0.5),
                "density": ("Density", 0.5),
                "realization": ("Realization", 0.5),
                "chaos": ("Entropy", 0.5),
                "stable": ("Entropy", -0.3)
            },
            "adaptive_learning_rate_factor": 0.01,
            "adaptive_threshold_factor": 0.05
        }

    def _load_config_from_file(self, config_file):
        with open(config_file, 'r') as f:
            user_config = json.load(f)
            self.config.update(user_config)

    def get(self, key, default=None):
        return self.config.get(key, default)


class Agent:
    def __init__(self, name, dimension_focus, config: ManifoldConfig):
        self.name = name
        self.dimension_focus = dimension_focus
        self.config = config
        self.mass = self.config.get("agent_initial_masses").get(name, self.config.get("initial_mass"))
        self.will_strength = self.mass * self.config.get("will_strength_multiplier")

    def exert_will(self, state_vector, user_input, other_agents):
        self.will_strength = self.mass * self.config.get("will_strength_multiplier")
        if self.dimension_focus in state_vector:
            delta = self.will_strength * (1.0 - state_vector[self.dimension_focus])
            state_vector[self.dimension_focus] += delta
        return state_vector

    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        pass


class CreativityAgent(Agent):
    def __init__(self, config: ManifoldConfig):
        super().__init__("Creativity_Agent", "Creativity", config)
    def exert_will(self, state_vector, user_input, other_agents):
        state_vector = super().exert_will(state_vector, user_input, other_agents)
        if state_vector["Logic"] > 0.7:
            reduction = 0.05 * self.mass
            state_vector["Logic"] = max(0.0, state_vector["Logic"] - reduction)
        return state_vector
    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        if target_agent.name == "Entropy_Agent" and current_state_vector["Entropy"] < 0.3:
            target_agent.mass += 0.01
            communication_buffer.append(f"[{self.name}] Praised {target_agent.name} for low Entropy.")
        if target_agent.name == "Logic_Agent" and current_state_vector["Logic"] > 0.8:
            target_agent.mass -= 0.005
            communication_buffer.append(f"[{self.name}] Critiqued {target_agent.name} for high Logic.")

class LogicAgent(Agent):
    def __init__(self, config: ManifoldConfig):
        super().__init__("Logic_Agent", "Logic", config)
    def exert_will(self, state_vector, user_input, other_agents):
        state_vector = super().exert_will(state_vector, user_input, other_agents)
        if state_vector["Entropy"] > 0.4:
            reduction = 0.1 * self.mass
            state_vector["Entropy"] = max(0.0, state_vector["Entropy"] - reduction)
        return state_vector
    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        if target_agent.name == "Realization_Agent" and current_state_vector["Realization"] > 0.6:
            target_agent.mass += 0.02
            communication_buffer.append(f"[{self.name}] Praised {target_agent.name} for high Realization.")
        if target_agent.name == "Entropy_Agent" and current_state_vector["Entropy"] > 0.6:
            target_agent.mass -= 0.01
            communication_buffer.append(f"[{self.name}] Critiqued {target_agent.name} for high Entropy.")

class DensityAgent(Agent):
    def __init__(self, config: ManifoldConfig):
        super().__init__("Density_Agent", "Density", config)
    def exert_will(self, state_vector, user_input, other_agents):
        state_vector = super().exert_will(state_vector, user_input, other_agents)
        if state_vector["Density"] > 0.6:
            push = 0.08 * self.mass
            state_vector["Realization"] = min(1.0, state_vector["Realization"] + push)
        return state_vector
    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        if target_agent.name == "Logic_Agent":
            target_agent.mass += 0.005
            communication_buffer.append(f"[{self.name}] Supported {target_agent.name} to maintain structure.")

class RealizationAgent(Agent):
    def __init__(self, config: ManifoldConfig):
        super().__init__("Realization_Agent", "Realization", config)
    def exert_will(self, state_vector, user_input, other_agents):
        state_vector = super().exert_will(state_vector, user_input, other_agents)
        if state_vector["Entropy"] > 0.7:
            state_vector["Entropy"] -= 0.05 * self.mass
        return state_vector
    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        if current_state_vector["Realization"] > 0.8:
            target_agent.mass += 0.005
            communication_buffer.append(f"[{self.name}] Rewarded {target_agent.name} for contributing to Realization.")

class EntropyAgent(Agent):
    def __init__(self, config: ManifoldConfig):
        super().__init__("Entropy_Agent", "Entropy", config)
    def exert_will(self, state_vector, user_input, other_agents):
        state_vector = super().exert_will(state_vector, user_input, other_agents)
        if state_vector["Creativity"] < 0.5:
            state_vector["Creativity"] += 0.03 * self.mass
        return state_vector
    def critique_and_modify(self, target_agent, current_state_vector, communication_buffer):
        if random.random() > 0.9:
            mass_change = (random.random() - 0.5) * 0.02
            target_agent.mass += mass_change
            communication_buffer.append(f"[{self.name}] Randomly influenced {target_agent.name} by {mass_change:.3f}.")

class AutonomousManifoldV3:
    def __init__(self, config_file=None, persistence_file="manifold_state.json"):
        self.name = "Unified_Manifold_v3"
        self.persistence_file = persistence_file
        self.config = ManifoldConfig(config_file)
        self.dimensions = list(self.config.get("q_weights").keys())
        self.q_weights = self.config.get("q_weights")
        self.destructive_coordinates = self.config.get("destructive_coordinates")
        self.learning_rate = self.config.get("learning_rate")
        self.target_state = self.config.get("target_state")
        self.agents = [
            CreativityAgent(self.config), LogicAgent(self.config),
            DensityAgent(self.config), RealizationAgent(self.config),
            EntropyAgent(self.config)
        ]
        self.communication_buffer = []
        self.load_state()

    def load_state(self):
        if os.path.exists(self.persistence_file):
            try:
                with open(self.persistence_file, 'r') as f:
                    data = json.load(f)
                    for agent in self.agents:
                        if agent.name in data.get("agent_masses", {}):
                            agent.mass = data["agent_masses"][agent.name]
            except Exception: pass

    def save_state(self):
        data = {"agent_masses": {agent.name: agent.mass for agent in self.agents}}
        try:
            with open(self.persistence_file, 'w') as f:
                json.dump(data, f)
        except Exception: pass

    def calculate_q_score(self, state_vector):
        q_score = sum((1.0 - state_vector[dim]) * weight if dim == "Entropy" else state_vector[dim] * weight for dim, weight in self.q_weights.items())
        q_score += 0.26 + (sum(a.mass for a in self.agents) * 0.05)
        return min(1.0, q_score)

    def _map_to_manifold(self, input_data):
        state_vector = {"Creativity": 0.2, "Logic": 0.2, "Density": 0.2, "Realization": 0.2, "Entropy": 0.4}
        user_state = input_data.get("user_state", "").lower()
        for kw, (dim, val) in self.config.get("keyword_influences").items():
            if kw in user_state: state_vector[dim] = min(1.0, max(0.0, state_vector[dim] + val))
        external = input_data.get("external_input", {})
        for dim, val in external.items():
            if dim in state_vector: state_vector[dim] = min(1.0, max(0.0, state_vector[dim] + val))
        return state_vector

    def _apply_destructive_interference(self, state_vector, current_q_score):
        logs = []
        entropy_threshold = self.config.get("destructive_coordinates")["entropy_collapse"] - (current_q_score * self.config.get("adaptive_threshold_factor"))
        logic_threshold = self.config.get("destructive_coordinates")["logic_override"] + (current_q_score * self.config.get("adaptive_threshold_factor"))
        density_threshold = self.config.get("destructive_coordinates")["density_singularity"] + (current_q_score * self.config.get("adaptive_threshold_factor"))
        if state_vector["Entropy"] >= entropy_threshold:
            state_vector["Creativity"] *= 0.2; state_vector["Logic"] *= 0.2; logs.append("Entropy Collapse")
        if state_vector["Logic"] >= logic_threshold:
            state_vector["Creativity"] = 0.0; logs.append("Logic Override")
        if state_vector["Density"] >= density_threshold:
            state_vector["Realization"] = 1.0; state_vector["Creativity"] = state_vector["Logic"] = state_vector["Entropy"] = 0.0; logs.append("Density Singularity")
        return state_vector, logs

    def _train_manifold(self, current_state, current_q_score):
        adaptive_lr = max(0.01, self.learning_rate * (1.0 - current_q_score * self.config.get("adaptive_learning_rate_factor")))
        for dim in self.dimensions:
            current_state[dim] = min(1.0, max(0.0, current_state[dim] + adaptive_lr * (self.target_state[dim] - current_state[dim])))
        return current_state

    def process(self, initial_input, iterations=5, training_mode=True, external_data=None):
        current_state = self._map_to_manifold(initial_input)
        if external_data and "external_influence" in external_data:
            for dim, val in external_data["external_influence"].items():
                if dim in current_state: current_state[dim] = min(1.0, max(0.0, current_state[dim] + val * 0.1))
        for i in range(iterations):
            self.communication_buffer = []
            for agent in self.agents:
                current_state = agent.exert_will(current_state, initial_input, self.agents)
                for other in self.agents:
                    if agent != other: agent.critique_and_modify(other, current_state, self.communication_buffer)
            current_q_score = self.calculate_q_score(current_state)
            current_state, logs = self._apply_destructive_interference(current_state, current_q_score)
            if training_mode: current_state = self._train_manifold(current_state, current_q_score)
            if current_q_score >= 0.95: break
        self.save_state()
        return {"q_score": self.calculate_q_score(current_state), "state_vector": current_state, "agent_masses": {a.name: round(a.mass, 3) for a in self.agents}}



In [ ]:
%%writefile hybrid_solver.py
"""
hybrid_solver.py — A Neuro-Symbolic Hybrid ARC Solver.
"""
import os
import sys
from typing import Optional, Tuple, Dict, List
import random
import numpy as np
import torch
import torch.nn as nn

SEED = 42
N_COLORS = 10
OOB_INDEX = 10
NEURAL_EPOCHS = 1000
NEURAL_LR = 0.01
NEURAL_EMB_DIM = 32

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

def set_deterministic(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class SymbolicSolver:
    def __init__(self):
        self.geom_transforms = {
            "identity": lambda x: np.ascontiguousarray(x.copy()),
            "rot90_1": lambda x: np.ascontiguousarray(np.rot90(x, 1)),
            "rot90_2": lambda x: np.ascontiguousarray(np.rot90(x, 2)),
            "rot90_3": lambda x: np.ascontiguousarray(np.rot90(x, 3)),
            "flip_lr": lambda x: np.ascontiguousarray(np.fliplr(x)),
            "flip_ud": lambda x: np.ascontiguousarray(np.flipud(x)),
            "transpose": lambda x: np.ascontiguousarray(x.T),
            "anti_transpose": lambda x: np.ascontiguousarray(np.rot90(x, 2).T)
        }

    def find_consistent_geom_transform(self, train_pairs: List[dict]) -> Optional[str]:
        for name, func in self.geom_transforms.items():
            solved_all = True
            for pair in train_pairs:
                inp, out = np.array(pair["input"]), np.array(pair["output"])
                if not np.array_equal(func(inp), out):
                    solved_all = False; break
            if solved_all: return name
        return None

    def find_consistent_geom_plus_color(self, train_pairs: List[dict]) -> Optional[Tuple[str, Dict[int, int]]]:
        for geom_name, geom_func in self.geom_transforms.items():
            mapping, reverse_mapping = {}, {}
            valid = True
            for pair in train_pairs:
                inp, out = np.array(pair["input"]), np.array(pair["output"])
                transformed = geom_func(inp)
                if transformed.shape != out.shape:
                    valid = False; break
                for src, dst in zip(transformed.flat, out.flat):
                    src, dst = int(src), int(dst)
                    if (src in mapping and mapping[src] != dst) or (dst in reverse_mapping and reverse_mapping[dst] != src):
                        valid = False; break
                    mapping[src], reverse_mapping[dst] = dst, src
                if not valid: break
            if valid: return geom_name, mapping
        return None

    def find_consistent_color_mapping(self, train_pairs: List[dict]) -> Optional[Dict[int, int]]:
        mapping, reverse_mapping = {}, {}
        for pair in train_pairs:
            inp, out = np.array(pair["input"]), np.array(pair["output"])
            if inp.shape != out.shape: return None
            for src, dst in zip(inp.flat, out.flat):
                src, dst = int(src), int(dst)
                if (src in mapping and mapping[src] != dst) or (dst in reverse_mapping and reverse_mapping[dst] != src):
                    return None
                mapping[src], reverse_mapping[dst] = dst, src
        return mapping

def grid_to_tabular(inp_grid: np.ndarray, out_grid: np.ndarray = None):
    H, W = inp_grid.shape
    ys, xs = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    y_norm = (ys.flatten() / max(1, H - 1)) - 0.5
    x_norm = (xs.flatten() / max(1, W - 1)) - 0.5
    num_feats = np.stack([y_norm, x_norm], axis=-1)
    padded = np.pad(inp_grid, pad_width=1, mode='constant', constant_values=OOB_INDEX)
    cat_feats = []
    for r in range(H):
        for c in range(W):
            pr, pc = r + 1, c + 1
            pixel_neighborhood = [padded[pr, pc]]
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0: continue
                    pixel_neighborhood.append(padded[pr + dr, pc + dc])
            cat_feats.append(pixel_neighborhood)
    num_tensor = torch.tensor(num_feats, dtype=torch.float32)
    cat_tensor = torch.tensor(cat_feats, dtype=torch.long)
    targets_tensor = torch.tensor(out_grid.flatten(), dtype=torch.long) if out_grid is not None else None
    return num_tensor, cat_tensor, targets_tensor

class TabularCoordNet(nn.Module):
    def __init__(self, cat_cardinality: int = 11, emb_dim: int = 32):
        super().__init__()
        self.shared_emb = nn.Embedding(cat_cardinality, emb_dim)
        total_input_dim = 2 + (9 * emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(total_input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, N_COLORS)
        )
    def forward(self, num_x, cat_x):
        embedded_cats = self.shared_emb(cat_x)
        embedded_flat = embedded_cats.view(embedded_cats.size(0), -1)
        x = torch.cat([num_x, embedded_flat], dim=-1)
        return self.mlp(x)

class HybridARCSolver:
    def __init__(self, verbose: bool = False, seed: int = SEED):
        self.verbose, self.seed = verbose, seed
        set_deterministic(seed)
        self.symbolic_solver = SymbolicSolver()

    def log(self, msg: str):
        if self.verbose: print(msg)

    def solve(self, task: dict) -> List[np.ndarray]:
        train_pairs = task["train"]
        test_inputs = [np.array(test["input"]) for test in task["test"]]
        predictions = []

        # Stage 1
        rule = self.symbolic_solver.find_consistent_geom_transform(train_pairs)
        if rule:
            self.log(f"Stage 1a: {rule}")
            return [self.symbolic_solver.geom_transforms[rule](ti) for ti in test_inputs]
        
        gc_rule = self.symbolic_solver.find_consistent_geom_plus_color(train_pairs)
        if gc_rule:
            rule, cmap = gc_rule
            self.log(f"Stage 1b: {rule} + {cmap}")
            for ti in test_inputs:
                tr = self.symbolic_solver.geom_transforms[rule](ti)
                out = tr.copy()
                for s, d in cmap.items(): out[tr == s] = d
                predictions.append(out)
            return predictions

        # Stage 2
        lookup, consistent = {}, True
        for pair in train_pairs:
            _, cat_t, tgt_t = grid_to_tabular(np.array(pair["input"]), np.array(pair["output"]))
            for crow, tgt in zip(cat_t.tolist(), tgt_t.tolist()):
                k = tuple(crow)
                if k in lookup and lookup[k] != tgt: consistent = False; break
                lookup[k] = tgt
            if not consistent: break
        
        if consistent:
            all_cov = True
            for ti in test_inputs:
                _, tcat, _ = grid_to_tabular(ti)
                if any(tuple(r) not in lookup for r in tcat.tolist()): all_cov = False; break
            if all_cov:
                self.log("Stage 2a")
                for ti in test_inputs:
                    _, tcat, _ = grid_to_tabular(ti)
                    predictions.append(np.array([lookup[tuple(r)] for r in tcat.tolist()]).reshape(ti.shape))
                return predictions

        # Stage 2b
        self.log("Stage 2b")
        all_num, all_cat, all_tgt = [], [], []
        for pair in train_pairs:
            n, c, t = grid_to_tabular(np.array(pair["input"]), np.array(pair["output"]))
            all_num.append(n); all_cat.append(c); all_tgt.append(t)
        t_num, t_cat, t_tgt = torch.cat(all_num), torch.cat(all_cat), torch.cat(all_tgt)
        net = TabularCoordNet(emb_dim=NEURAL_EMB_DIM)
        opt = torch.optim.Adam(net.parameters(), lr=NEURAL_LR)
        crit = nn.CrossEntropyLoss()
        for _ in range(NEURAL_EPOCHS):
            opt.zero_grad(); crit(net(t_num, t_cat), t_tgt).backward(); opt.step()
        
        net.eval()
        with torch.no_grad():
            for ti in test_inputs:
                n, c, _ = grid_to_tabular(ti)
                predictions.append(torch.argmax(net(n, c), -1).numpy().reshape(ti.shape))
        return predictions

if __name__ == "__main__":
    set_deterministic(SEED)
    solver = HybridARCSolver(verbose=True)
    
    # Test 3
    context_task = {
        "train": [
            {"input": [[1, 2, 0], [0, 1, 0]], "output": [[4, 2, 0], [0, 4, 0]]},
            {"input": [[1, 0, 0], [0, 0, 2]], "output": [[1, 0, 0], [0, 0, 2]]}
        ],
        "test": [
            {"input": [[0, 1, 2], [1, 0, 0]]}
        ]
    }
    pred = solver.solve(context_task)[0]
    expected = np.array([[0, 4, 2], [1, 0, 0]])
    if np.array_equal(pred, expected): print("Test 3 Passed")
    else: print(f"Test 3 Failed. Pred:\n{pred}")


In [ ]:
%%writefile financial_data_utils.py
import json, torch, os, pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader
class FinancialAssetDataset(Dataset):
    def __init__(self, csv_path):
        self.samples = []
        if not os.path.exists(csv_path): return
        df = pd.read_csv(csv_path)
        for _, g in df.groupby(['adsh', 'ddate']):
            a, l = g[g['tag'] == 'Assets']['value'], g[g['tag'] == 'Liabilities']['value']
            if not a.empty and not l.empty:
                self.samples.append((torch.tensor(self._to_8bit(a.iloc[0]), dtype=torch.float32), torch.tensor(self._to_8bit(l.iloc[0]), dtype=torch.float32)))
    def _to_8bit(self, val):
        if val <= 0: return [0]*8
        q = int(np.clip((np.log10(val) - 6) / 6 * 255, 0, 255))
        return [int(b) for b in format(q, '08b')]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]
def get_financial_dataloader(path, batch_size=32):
    ds = FinancialAssetDataset(path)
    return DataLoader(ds, batch_size=batch_size, shuffle=True) if len(ds) > 0 else [(torch.randn(batch_size, 8), torch.randint(0, 2, (batch_size, 8)).float())]


In [ ]:
%%writefile extract_financials.py
import json, os, pandas as pd
from io import StringIO
def extract_from_json(json_path):
    with open(json_path, 'r') as f: data = json.load(f)
    num_df = pd.read_csv(StringIO(data['num.txt']), sep='\t')
    sub_df = pd.read_csv(StringIO(data['sub.txt']), sep='\t')
    target_tags = ['Assets', 'Liabilities', 'AssetsCurrent', 'LiabilitiesCurrent']
    merged = pd.merge(num_df[num_df['tag'].isin(target_tags)], sub_df[['adsh', 'name', 'cik']], on='adsh')
    return merged
def main():
    all_data = []
    base_dir = 'kaggle_data/sec_financials'
    if not os.path.exists(base_dir): return
    for f in [f for f in os.listdir(base_dir) if f.endswith('.json')]:
        try: all_data.append(extract_from_json(os.path.join(base_dir, f)))
        except: pass
    if all_data: pd.concat(all_data, ignore_index=True).to_csv('extracted_assets_liabilities.csv', index=False)
if __name__ == "__main__": main()


In [ ]:
%%writefile populate_financial_manifold.py
import os, numpy as np, pandas as pd
from autonomous_manifold_v3 import AutonomousManifoldV3
def project_to_1024(vector, masses):
    base = np.array(list(vector.values()) + list(masses.values()))
    expanded = np.tile(base, (1024 // len(base)) + 1)[:1024]
    return (expanded + 0.1 * np.sin(np.linspace(0, 10, 1024) * expanded.sum())).reshape(32, 32)
def main():
    if not os.path.exists('extracted_assets_liabilities.csv'): return
    df = pd.read_csv('extracted_assets_liabilities.csv')
    df['ddate'] = pd.to_datetime(df['ddate'])
    pivoted = df.loc[df.groupby(['adsh', 'tag'])['ddate'].idxmax()].pivot(index='adsh', columns='tag', values='value').dropna(subset=['Assets', 'Liabilities'], how='all')
    pivoted['ratio'] = pivoted['Assets'] / pivoted['Liabilities'].replace(0, 1)
    out_dir = 'kaggle_data/financial_manifold'
    os.makedirs(out_dir, exist_ok=True)
    manifold = AutonomousManifoldV3(persistence_file="financial_manifold_state.json")
    for count, (adsh, row) in enumerate(pivoted.sort_values(by='Assets', ascending=False).head(100).iterrows()):
        ext = {"Logic": min(1.0, row['Assets'] / 1e12), "Realization": min(1.0, row['ratio'] / 5.0), "Density": min(1.0, row.get('AssetsCurrent', 0) / (row['Assets'] + 1e-6))}
        res = manifold.process({"user_state": f"Financial evolution", "external_input": ext}, iterations=5)
        np.save(os.path.join(out_dir, f'weight_{count}.npy'), project_to_1024(res['state_vector'], res['agent_masses']))
        lib = np.tile(np.array(list(res['agent_masses'].values())), (1024 // len(res['agent_masses'])) + 1)[:1024].reshape(32, 32)
        np.save(os.path.join(out_dir, f'lib_{count}.npy'), lib)
if __name__ == "__main__": main()


In [ ]:
%%writefile train_financial_leverage.py
import torch, torch.nn as nn, torch.optim as optim, os
from sheaf_nn import DeepSheafNetwork
from weights_loader import ManifoldLoader
from financial_data_utils import get_financial_dataloader
def train(dry_run=False, num_epochs=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DeepSheafNetwork(8, [(i, (i+1)%8) for i in range(8)], 32, 32, num_layers=8, layer_type='nca').to(device)
    try: model.load_from_manifold(ManifoldLoader(directory='kaggle_data/financial_manifold'))
    except: pass
    head = nn.Sequential(nn.Linear(32, 128), nn.LayerNorm(128), nn.ReLU(), nn.Linear(128, 8)).to(device)
    opt = optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=1e-4)
    crit = nn.BCEWithLogitsLoss()
    dl = get_financial_dataloader('extracted_assets_liabilities.csv', batch_size=64)
    for epoch in range(num_epochs if not dry_run else 1):
        model.train()
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            crit(head(model(x.unsqueeze(-1).repeat(1, 1, 32))).mean(1), y).backward()
            opt.step()
        print(f"Epoch {epoch} complete")
if __name__ == "__main__": train(dry_run=True)


In [ ]:
%%writefile arc_manifold_solver.py
import json
import numpy as np
from autonomous_manifold_v3 import AutonomousManifoldV3, ManifoldConfig

class ARCTransformers:
    @staticmethod
    def identity(grid): return grid.tolist()
    
    @staticmethod
    def rotate_90(grid): return np.rot90(grid, k=1).tolist()
    
    @staticmethod
    def rotate_180(grid): return np.rot90(grid, k=2).tolist()
    
    @staticmethod
    def rotate_270(grid): return np.rot90(grid, k=3).tolist()
    
    @staticmethod
    def flip_v(grid): return np.flipud(grid).tolist()
    
    @staticmethod
    def flip_h(grid): return np.fliplr(grid).tolist()
    
    @staticmethod
    def transpose(grid): return grid.T.tolist()

class ARCFeatureExtractorV2:
    @staticmethod
    def analyze_transformations(train_pairs):
        """Detects which standard transformations work for the training set."""
        possible_transforms = {
            "identity": ARCTransformers.identity,
            "rotate_90": ARCTransformers.rotate_90,
            "rotate_180": ARCTransformers.rotate_180,
            "rotate_270": ARCTransformers.rotate_270,
            "flip_v": ARCTransformers.flip_v,
            "flip_h": ARCTransformers.flip_h,
            "transpose": ARCTransformers.transpose
        }
        
        scores = {k: 0 for k in possible_transforms}
        
        for pair in train_pairs:
            in_grid = np.array(pair["input"])
            out_grid = np.array(pair["output"])
            
            for name, func in possible_transforms.items():
                try:
                    transformed = np.array(func(in_grid))
                    if transformed.shape == out_grid.shape and np.array_equal(transformed, out_grid):
                        scores[name] += 1
                except:
                    continue
                    
        return scores

    @staticmethod
    def extract(task_data):
        train_pairs = task_data.get("train", [])
        transform_scores = ARCFeatureExtractorV2.analyze_transformations(train_pairs)
        
        num_pairs = len(train_pairs) if train_pairs else 1
        max_score = max(transform_scores.values()) if transform_scores else 0
        
        features = {
            "Creativity": 0.3,
            "Logic": max_score / num_pairs,
            "Density": np.mean([np.count_nonzero(p["output"]) / np.array(p["output"]).size for p in train_pairs]) if train_pairs else 0.5,
            "Realization": 0.1,
            "Entropy": 0.1 if max_score == num_pairs else 0.8
        }
        
        return features, transform_scores

class ARCManifoldSolverV2:
    def __init__(self):
        self.manifold = AutonomousManifoldV3()
        
    def solve_task(self, task_id, task_data):
        features, transform_scores = ARCFeatureExtractorV2.extract(task_data)
        
        result = self.manifold.process(
            {"user_state": f"Solve ARC task {task_id}", "external_input": features},
            iterations=5,
            training_mode=True
        )
        
        sorted_transforms = sorted(transform_scores.items(), key=lambda x: x[1], reverse=True)
        best_transform_name = sorted_transforms[0][0] if sorted_transforms and sorted_transforms[0][1] > 0 else "identity"
        second_best_name = sorted_transforms[1][0] if len(sorted_transforms) > 1 and sorted_transforms[1][1] > 0 else "flip_v"
        
        test_inputs = task_data.get("test", [])
        predictions = []
        
        for test_in in test_inputs:
            grid = np.array(test_in["input"])
            attempt_1 = getattr(ARCTransformers, best_transform_name)(grid)
            attempt_2 = getattr(ARCTransformers, second_best_name)(grid)
            predictions.append({"attempt_1": attempt_1, "attempt_2": attempt_2})
            
        return predictions

def solve_arc_tasks(dataset_path, limit=100):
    if not os.path.exists(dataset_path):
        print(f"Dataset not found: {dataset_path}")
        return
    with open(dataset_path, "r") as f:
        challenges = json.load(f)
    solver = ARCManifoldSolverV2()
    submission = {}
    task_ids = list(challenges.keys())[:limit]
    for task_id in task_ids:
        print(f"Solving: {task_id}")
        submission[task_id] = solver.solve_task(task_id, challenges[task_id])
    for task_id in challenges:
        if task_id not in submission:
            test_inputs = challenges[task_id].get("test", [])
            submission[task_id] = [{"attempt_1": t["input"], "attempt_2": t["input"]} for t in test_inputs]
    with open("submission.json", "w") as f:
        json.dump(submission, f, indent=4)
    print(f"\n[SUCCESS] ARC Manifold Solver V2 generated submission.json for {len(submission)} tasks.")

if __name__ == "__main__":
    # solve_arc_tasks("/home/ubuntu/arc-agi-dataset/arc-agi_training_challenges.json")
    pass


## Population of Financial Manifold
Extracting assets and liabilities.

In [ ]:
!python3 extract_financials.py
!python3 populate_financial_manifold.py

## ARC Manifold Reasoning
Demonstrating autonomous reasoning on a sample ARC task.

In [ ]:
from arc_manifold_solver import ARCManifoldSolverV2
import json

sample_task = {
    "train": [{"input": [[1, 1], [0, 0]], "output": [[0, 0], [1, 1]]}],
    "test": [{"input": [[1, 0], [1, 0]]}]
}
solver = ARCManifoldSolverV2()
prediction = solver.solve_task("sample_0", sample_task)
print(f"Sample Prediction: {json.dumps(prediction, indent=2)}")

## Financial Asset-Liability Training

In [ ]:
from train_financial_leverage import train as train_financial
train_financial(dry_run=True)